# Optimized Random Forest

This notebook uses the final optimized settings and frozen prediction threshold for this model. It displays training, validation and test accuracy, optimization settings, and fitting diagnostics. The combined comparison remains in `FINAL_Four_Optimized_Models.ipynb`. Test results are reused-holdout estimates, not independent validation.

In [1]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
import joblib
from sklearn.base import clone
from IPython.display import display
root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'src/model_workflow.py').exists())
sys.path.insert(0, str(root/'src'))
from model_workflow import paths, load_features, matrix, TARGET, metrics
MODEL = 'random_forest'
folder = root/'outputs/final_four'
bundle = joblib.load(folder/f'{MODEL}.joblib')
model = bundle['pipeline']
metadata = bundle['metadata']
threshold = metadata['Threshold']
data, fingerprint = load_features(paths(root)['processed'])
assert fingerprint == metadata['fingerprint'], 'Data changed; revalidate optimization before use.'
print('Model:', MODEL)
print('Optimized variant:', metadata['Source'])
print('Frozen threshold:', threshold)
print('Selected overall:', metadata['selected'])

Model: random_forest
Optimized variant: random_forest
Frozen threshold: 0.425
Selected overall: True


## Optimized pipeline and settings

Preprocessing is part of the fitted pipeline. Hyperparameters were selected using training cross-validation; the operating threshold was selected on validation data.

In [2]:
display(model)
display(pd.DataFrame(list(model.named_steps['model'].get_params().items()), columns=['Parameter','Value']))

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('features', ...), ('preprocess', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('numeric', ...), ('country', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that

,Parameter,Value
0,bootstrap,True
1,ccp_alpha,0.0
2,class_weight,None
3,criterion,gini
4,max_depth,8
5,max_features,0.7
6,max_leaf_nodes,None
7,max_samples,1.0
8,min_impurity_decrease,0.0
9,min_samples_leaf,60


### Why the final settings were retained

max_depth=8 and min_samples_leaf=60 constrain individual trees; max_features=0.7 introduces feature subsampling, while 350 trees average predictions across bootstrap samples. max_samples=1.0 retains full-size bootstrap draws. This retained variant passed fitting screens and achieved the highest final validation accuracy (68.63%). The chosen values balance validation performance with the measured fitting gaps rather than maximizing training accuracy.


## Hyperparameter tuning evidence (loaded historical results)

The final fitted hyperparameters are displayed above. The table below loads the retained **updated optimization search**, including the tested parameter distributions, best mean CV score, selected trial and its three fold scores. These later trials were compared with earlier incumbent variants; the final bundle did not necessarily come from the updated search. The original incumbent per-trial search files are no longer in the clean package, so the later trial must not be described as the exact origin of the final model.

Search: RandomizedSearchCV, at most 32 configurations, random_state=73; three customer-grouped folds, shuffle=True, random_state=42. All CV accuracies here use threshold 0.50. The training-only selection rule requires a train–CV gap ≤5 percentage points, CV improvement ≥3 points over the majority baseline, and CV within 1 point of the search best; near-ties within 0.2 points prefer a smaller gap. Thus the selected trial can differ from the highest raw CV score.


In [3]:
from tuning_workflow import historical_search, GRIDS, load_cv, frozen_cv, tune
from accuracy_experiments import candidates, make_pipeline, preparation
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import ParameterGrid

# Parameter distributions for the retained historical UPDATED search.
historical_name = 'optimized_' + MODEL
historical_spaces = {name: space for name, pipeline, space in candidates()}
historical_spaces['optimized_gradient_boosting'] = {
    'prepare': [preparation(True), preparation(False)],
    'model__n_estimators': [60, 120, 240],
    'model__learning_rate': [.025, .05, .1],
    'model__max_depth': [1, 2, 3],
    'model__min_samples_leaf': [40, 80, 120],
    'model__subsample': [.7, 1.0],
}
display(pd.DataFrame([{'Parameter': k, 'Historical distribution': str(v)}
                      for k, v in historical_spaces[historical_name].items()]))
historical_results, historical_index = historical_search(MODEL, root)
trial = historical_results.iloc[historical_index]
print('LOADED historical updated-search evidence; no tuning performed by this cell.')
print('Tested configurations:', len(historical_results))
print('Best historical search CV accuracy:', f'{historical_results.mean_test_accuracy.max():.4%}')
print('Rule-selected historical trial CV accuracy:', f'{trial.mean_test_accuracy:.4%}')
display(trial[[c for c in historical_results.columns if c.startswith('param_')]].to_frame('Selected historical trial'))
display(pd.DataFrame({'Fold': [1, 2, 3],
    'Historical selected-trial CV accuracy': [trial[f'split{i}_test_accuracy'] for i in range(3)],
    'Historical selected-trial training accuracy': [trial[f'split{i}_train_accuracy'] for i in range(3)]}))
family_records = pd.read_csv(root/'outputs/evidence/optimization/validation_comparison.csv')
variant_names = [MODEL, 'optimized_'+MODEL, 'updated_optimized_'+MODEL]
display(family_records[family_records.Model.isin(variant_names)][
    ['Model', 'Validation accuracy', 'CV accuracy', 'Threshold', 'Eligible']])
print('Final retained source variant:', metadata['Source'])
print('The final variant was chosen by eligible validation accuracy and the frozen-threshold audit, not by test rank.')


,Parameter,Historical distribution
0,prepare,"[ColumnTransformer(transformers=[('numeric',\n..."
1,model__max_depth,"[3, 4, 6, 8]"
2,model__min_samples_leaf,"[20, 40, 80]"
3,model__max_features,"['sqrt', 0.7, 1.0]"
4,model__class_weight,"[None, 'balanced']"


LOADED historical updated-search evidence; no tuning performed by this cell.
Tested configurations: 32
Best historical search CV accuracy: 69.3339%
Rule-selected historical trial CV accuracy: 69.2340%


,Selected historical trial
param_prepare,"ColumnTransformer(transformers=[('numeric',\n ..."
param_model__min_samples_leaf,80
param_model__max_features,1.0
param_model__max_depth,6
param_model__class_weight,NaN


,Fold,Historical selected-trial CV accuracy,Historical selected-trial training accuracy
0,1,0.729483,0.695751
1,2,0.686930,0.713202
2,3,0.660606,0.723404


,Model,Validation accuracy,CV accuracy,Threshold,Eligible
2,random_forest,0.686301,0.695878,0.425,True
8,optimized_random_forest,0.683562,0.696386,0.425,True
15,updated_optimized_random_forest,0.684932,0.692340,0.425,True


Final retained source variant: random_forest
The final variant was chosen by eligible validation accuracy and the frozen-threshold audit, not by test rank.


## Cross-validation of the final model settings

The saved evidence below was **newly computed during this revision**, by cloning the final pipeline and fitting it separately in each of three StratifiedGroupKFold training folds (seed 42). At ordinary notebook execution it is **loaded**, not recomputed. Customer IDs never overlap across the train/held-out sides of a fold; imputation, encoding and scaling are refitted inside each clone. Fold sizes and customer counts are included. The CSV and provenance JSON identify the data fingerprint, bundle hash and software versions.

Both default 0.50 and frozen operating thresholds are shown. Their means are unweighted averages across three folds, not pooled accuracy. This seed-42 diagnostic differs from the older seed-137 frozen-threshold audit; do not mix their scores. Set RECOMPUTE_CV=True to recompute these diagnostics without changing the final fitted bundles.


In [4]:
RECOMPUTE_CV = False
if RECOMPUTE_CV:
    cv_folds = frozen_cv(MODEL, root)
    print('NEWLY COMPUTED: clones refitted on three training folds.')
else:
    cv_folds, cv_provenance = load_cv(MODEL, root)
    print('LOADED: previously computed final-settings CV evidence; no training in this cell.')
    print(cv_provenance)
display(cv_folds)
cv_summary = cv_folds.groupby('threshold')[['train_accuracy', 'cv_accuracy']].mean()
cv_summary['train_cv_gap'] = cv_summary.train_accuracy - cv_summary.cv_accuracy
display(cv_summary)
print('Final settings mean CV accuracy (0.50):', f"{cv_summary.loc[0.5, 'cv_accuracy']:.4%}")


LOADED: previously computed final-settings CV evidence; no training in this cell.
{'model': 'random_forest', 'fingerprint': '4d1cb943f51e641ec287113ab684d7345a7c84bf1b93f8458088adcdec90b73d', 'bundle_sha256': '2ee60346051d5819e5b277dfb1e6d86b9077973b8da662259a8d866e64186bba', 'python': '3.12.14', 'sklearn': '1.9.1', 'folds': 3, 'fold_seed': 42, 'status': 'Newly computed diagnostic; clones fitted only on training folds', 'limitation': 'Not nested CV or independent temporal validation; holdouts were reused during development.'}


,fold,threshold,training_rows,heldout_rows,training_customers,heldout_customers,train_accuracy,cv_accuracy
0,1,0.425,1318,658,749,373,0.702580,0.705167
1,1,0.500,1318,658,749,373,0.703338,0.724924
2,2,0.425,1318,658,748,374,0.709408,0.683891
3,2,0.500,1318,658,748,374,0.720030,0.694529
4,3,0.425,1316,660,747,375,0.719605,0.666667
5,3,0.500,1316,660,747,375,0.731763,0.668182


,train_accuracy,cv_accuracy,train_cv_gap
threshold,,,
0.425,0.710531,0.685241,0.025289
0.500,0.718377,0.695878,0.022499


Final settings mean CV accuracy (0.50): 69.5878%


## Reproduce training and tuning (optional)

The clean-package rerun below uses the supplied final pipeline as an **unfitted template**: sklearn.clone discards all learned state. The displayed rerun grid includes the published setting and nearby alternatives. It fixes the final feature/preprocessing design; it is an explicit new reproducible experiment, not an exact replay of every historical feature/search trial. It requires only the included snapshot CSV, final bundle and runtime helpers.

RUN_TUNING=False loads no new candidate. Set it to True for up to 32 randomized configurations with three grouped folds. Candidate selection uses training CV, then June validation selects its threshold. No test scores enter the rerun. Results, chosen settings, fold scores and a candidate bundle are written to outputs/reproduction/<model>/; published final models stay unchanged. A candidate failing screening is recorded as rejected. A future promotion would require repeated and threshold-matched audits plus independent future-period evidence.


In [5]:
display(pd.DataFrame([{'Parameter': k, 'Rerun grid': str(v)} for k, v in GRIDS[MODEL].items()]))
RUN_TUNING = False
if RUN_TUNING:
    rerun_results, rerun_record = tune(MODEL, root, n_iter=32)
    print('NEWLY COMPUTED tuning experiment; original published model unchanged.')
    print(json.dumps(rerun_record, indent=2))
    display(rerun_results.sort_values('mean_test_accuracy', ascending=False).head())
else:
    print('Tuning not run. Set RUN_TUNING=True or use:')
    print(f'python src/tuning_workflow.py --model {MODEL} --mode tune --n-iter 32')


,Parameter,Rerun grid
0,model__n_estimators,"[200, 350]"
1,model__max_depth,"[3, 4, 6, 8]"
2,model__min_samples_leaf,"[20, 40, 60, 80]"
3,model__max_features,"['sqrt', 0.7, 1.0]"


Tuning not run. Set RUN_TUNING=True or use:
python src/tuning_workflow.py --model random_forest --mode tune --n-iter 32


## Validation limitations

Grouped CV estimates performance on held-out customers **within the training period**. It does not reproduce seasonal change or replace independent temporal validation. Customers may recur between the March/April training snapshots, June validation and August test snapshots; grouping applies within CV, not across those temporal partitions. Search CV is reused to choose settings, so its best score is not an unbiased nested-CV estimate. Repeated diagnostics are not independent experiments.

Validation and test holdouts were inspected repeatedly during project development. Their reported scores are reused-holdout evidence. The validation-selected threshold and variant can overfit validation even when CV gaps are small. Snapshot-level caps and country grouping were calculated from past cohorts, not independently inside customer CV folds. A genuinely new future period is needed for independent confirmation.


## Optional training with the frozen settings

Set `RETRAIN=True` to refit the same optimized pipeline on the training split only. This does not rerun hyperparameter search, change thresholds, or overwrite the published model files.

In [6]:
RETRAIN = False
if RETRAIN:
    training = data[data.split.eq('train')]
    model = clone(bundle['pipeline']).fit(matrix(training), training[TARGET])
    print('Refitted frozen settings on training data only.')
else:
    print('Using the verified saved optimized model.')

Using the verified saved optimized model.


## Training and test accuracy

Every split uses the same frozen threshold. Test scores are for reporting only; do not adjust parameters from them.

In [7]:
rows = []
for split in ['train','validation','test']:
    part = data[data.split.eq(split)]
    probability = model.predict_proba(matrix(part))[:,1]
    result = metrics(MODEL, part[TARGET], probability, threshold)
    result['Split'] = split
    rows.append(result)
results = pd.DataFrame(rows).set_index('Split')
display(results[['Accuracy','Precision','Recall','F1','ROC-AUC','PR-AUC','Threshold']])
print(f"Training accuracy: {results.loc['train','Accuracy']:.2%}")
print(f"Test accuracy: {results.loc['test','Accuracy']:.2%}")
if not RETRAIN:
    expected = pd.read_csv(folder/'accuracy_summary.csv').set_index('Model').loc[MODEL]
    for split,column in [('train','Training accuracy'),('validation','Validation accuracy'),('test','Test accuracy')]:
        assert np.isclose(results.loc[split,'Accuracy'], expected[column])
    print('Verified against the final comparison table.')

,Accuracy,Precision,Recall,F1,ROC-AUC,PR-AUC,Threshold
Split,,,,,,,
train,0.715587,0.605473,0.710843,0.653941,0.791265,0.679118,0.425
validation,0.686301,0.665610,0.629685,0.647149,0.744751,0.682493,0.425
test,0.671241,0.556617,0.668852,0.607595,0.731184,0.592246,0.425


Training accuracy: 71.56%
Test accuracy: 67.12%
Verified against the final comparison table.


## Fitting checks

The CV gap below is from the saved optimized model audit using its frozen threshold. Positive gaps alone do not prove harmful overfitting. The 5-point limit is a screening heuristic; a small gap does not prove absence of underfitting. Historical CV diagnostics must be rerun if settings or training data change.

In [8]:
print(f"Train-validation gap: {100*(results.loc['train','Accuracy']-results.loc['validation','Accuracy']):.2f} percentage points")
print(f"Train-test gap: {100*(results.loc['train','Accuracy']-results.loc['test','Accuracy']):.2f} percentage points")
print(f"Saved threshold-matched CV gap: {100*metadata['Threshold-matched CV gap']:.2f} percentage points")
print('Final saved model passes current fitting screens; fresh future data is needed to confirm generalization.')

Train-validation gap: 2.93 percentage points
Train-test gap: 4.43 percentage points
Saved threshold-matched CV gap: 4.15 percentage points
Final saved model passes current fitting screens; fresh future data is needed to confirm generalization.
